In [8]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import roc_auc_score
from lifelines.utils import concordance_index
from sklearn.metrics import brier_score_loss

In [15]:
vals = {}

def partial_cindex(y_true, y_pred, group_mask):
    group_idx = np.where(group_mask)[0]

    y_true_i = y_true[group_idx][:, np.newaxis]
    y_pred_i = y_pred[group_idx][:, np.newaxis]

    y_true_j = y_true[np.newaxis, :]
    y_pred_j = y_pred[np.newaxis, :]

    # Ignore comparisons where i == j
    mask_diff = group_idx[:, np.newaxis] != np.arange(len(y_true))

    # Valid comparisons (different y_true)
    mask_valid = (y_true_i != y_true_j) & mask_diff

    # Concordant pairs
    concordant = (
        ((y_true_i <= y_true_j) & (y_pred_i <= y_pred_j)) |
        ((y_true_i >= y_true_j) & (y_pred_i >= y_pred_j))
    ) & mask_valid

    permissible = np.sum(mask_valid)
    concordant_count = np.sum(concordant) 

    return concordant_count / permissible if permissible > 0 else np.nan
    
vals['Mean Signed Error of IPCW on Censored Data'] = []
vals['Average TTE - TTC'] = []
vals['Model'] = []

vals['MAE'] = []
vals['MAE on Censored Data'] = []
vals['MAE on Uncensored Data'] = []

vals['Mean Signed Error'] = []
vals['Mean Signed Error on Censored Data'] = []
vals['Mean Signed Error on Uncensored Data'] = []

vals['C-Index on Censored Data'] = []
vals['C-Index on Uncensored Data'] = []

vals['Max - Avg TTE of Censored Data'] = []



all_models = ['icind',  'pcind', 'dhcind', 'proposed', 'proposed_sameweight', 'proposed_samecluster', 'proposed_assignMAX', 'oracle']
all_models_names = ['IPCW',   "Standard\nMargin", 'Standard', 'Proposed', 'Proposed Without Weight', 'Proposed Without Cluster', 'Proposed Assign Max', 'Oracle']

print(len(all_models), len(all_models_names))
for COUNT in range(1000):
    print(COUNT)
    suffix = 'COUNT%d'%(COUNT)
    
    X_test = joblib.load('/data4/meerak/onevar_data/X_test_%s.joblib'%suffix).reshape(-1, 1)
    y_test = joblib.load('/data4/meerak/onevar_data/y_test_%s.joblib'%suffix)
    binary_y_test = joblib.load('/data4/meerak/onevar_data/actual_binary_y_test%s.joblib'%suffix)

    X_train = joblib.load('/data4/meerak/onevar_data/X_train_%s.joblib'%suffix).reshape(-1, 1)
    orig_y_train = joblib.load('/data4/meerak/onevar_data/orig_y_train_%s.joblib'%suffix)
    y_train = joblib.load('/data4/meerak/onevar_data/y_train_%s.joblib'%suffix)
    binary_y_train = joblib.load('/data4/meerak/onevar_data/binary_y_train_%s.joblib'%suffix)

    X_val = joblib.load('/data4/meerak/onevar_data/X_val_%s.joblib'%suffix).reshape(-1, 1)
    y_val = joblib.load('/data4/meerak/onevar_data/y_val_%s.joblib'%suffix)
    binary_y_val = joblib.load('/data4/meerak/onevar_data/binary_y_val_%s.joblib'%suffix)
    
    medcens = np.mean(np.abs(orig_y_train - y_train))

    not_exist = 0
    try:
        for model, modelname in zip(all_models, all_models_names):     
            preds = joblib.load('/data4/meerak/onevar_test_preds/%s_y_test_pred_%s.joblib'%(model, suffix))
            if len(preds) != len(y_test):
                not_exist = 1
    except:
        not_exist = 1
    

    if not_exist == 0:
        for model, modelname in zip(all_models, all_models_names):
            uncensored_preds = joblib.load('/data4/meerak/onevar_test_preds/icind_y_test_pred_%s.joblib'%(suffix))
            mse_mu_ec = np.mean(uncensored_preds[binary_y_test == 0] - y_test[binary_y_test == 0])
            preds = joblib.load('/data4/meerak/onevar_test_preds/%s_y_test_pred_%s.joblib'%(model, suffix))


            vals['Mean Signed Error of IPCW on Censored Data'].append(mse_mu_ec)
            vals['Average TTE - TTC'].append(np.mean(orig_y_train[binary_y_train == 0] - y_train[binary_y_train == 0]))
            vals['Model'].append(modelname)


            vals['MAE'].append(np.mean(np.abs(preds - y_test)))
            vals['MAE on Censored Data'].append(np.mean(np.abs(preds[binary_y_test == 0] - y_test[binary_y_test == 0])))
            vals['MAE on Uncensored Data'].append(np.mean(np.abs(preds[binary_y_test == 1] - y_test[binary_y_test == 1])))
            vals['Max - Avg TTE of Censored Data'].append(max(orig_y_train[binary_y_train == 0]) - np.mean(orig_y_train[binary_y_train == 0]))

            vals['Mean Signed Error'].append(np.mean(preds - y_test))
            vals['Mean Signed Error on Censored Data'].append(np.mean(preds[binary_y_test == 0] - y_test[binary_y_test == 0]))
            vals['Mean Signed Error on Uncensored Data'].append(np.mean(preds[binary_y_test == 1] - y_test[binary_y_test == 1]))

            vals['C-Index on Censored Data'].append(
                partial_cindex(y_test, preds, binary_y_test == 0)
            )
            
            vals['C-Index on Uncensored Data'].append(
                partial_cindex(y_test, preds, binary_y_test == 1)
            )
            


8 8
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
27

In [3]:
import pandas as pd
df = pd.DataFrame(vals)

In [4]:
df

,Mean Signed Error of IPCW on Censored Data,Average TTE - TTC,Model,MAE,MAE on Censored Data,MAE on Uncensored Data,Mean Signed Error,Mean Signed Error on Censored Data,Mean Signed Error on Uncensored Data,C-Index on Censored Data,C-Index on Uncensored Data,Max - Avg TTE of Censored Data
0,7.318140,10.820037,IPCW,5.798140,8.105743,1.357018,4.753749,7.318140,-0.181579,0.499757,0.918651,11.686920
1,7.318140,10.820037,Standard\nMargin,2.361728,2.798997,1.520175,-1.318536,-1.651322,-0.678070,0.833409,0.922673,11.686920
2,7.318140,10.820037,Standard,7.265447,10.353236,1.322807,6.706359,10.259344,-0.131579,0.481992,0.915777,11.686920
3,7.318140,10.820037,Proposed,2.280744,2.794895,1.291228,-1.424715,-2.164084,-0.001754,0.903245,0.915722,11.686920
4,7.318140,10.820037,Proposed Without Weight,5.789142,8.106199,1.329825,4.933113,7.318596,0.342105,0.500000,0.921069,11.686920
...,...,...,...,...,...,...,...,...,...,...,...,...
7379,7.098783,7.775567,Proposed,0.801140,1.072097,0.318030,-0.476005,-0.743446,0.000835,0.956090,0.986743,10.673299
7380,7.098783,7.775567,Proposed Without Weight,5.241452,7.884363,0.529215,4.819136,7.646536,-0.222037,0.643192,0.977194,10.673299
7381,7.098783,7.775567,Proposed Without Cluster,2.558188,3.686798,0.545910,-2.408218,-3.666199,-0.165275,0.891361,0.964890,10.673299
7382,7.098783,7.775567,Proposed Assign Max,0.894121,1.143727,0.449082,0.504799,0.790730,-0.005008,0.942497,0.979249,10.673299


In [14]:
for method in ['Proposed', 'Standard\nMargin', 'Standard', 'IPCW']:
    focus = df[df['Model'] == method]['C-Index on Censored Data']
    print(method, '%0.2f [%0.2f, %0.2f]'%(np.percentile(focus, 50), np.percentile(focus, 2.5), np.percentile(focus, 97.5)))

Proposed 0.92 [0.67, 0.98]
Standard
Margin 0.64 [0.44, 0.93]
Standard 0.53 [0.45, 0.78]
IPCW 0.52 [0.42, 0.76]


In [13]:
for method in ['Proposed', 'Standard\nMargin', 'Standard', 'IPCW']:
    focus = df[df['Model'] == method]['C-Index on Uncensored Data']
    print(method, '%0.2f [%0.2f, %0.2f]'%(np.percentile(focus, 50), np.percentile(focus, 2.5), np.percentile(focus, 97.5)))

Proposed 0.95 [0.64, 0.99]
Standard
Margin 0.95 [0.50, 0.99]
Standard 0.95 [0.50, 0.99]
IPCW 0.95 [0.54, 0.99]
